# 03 — Local tracking

Purpose: controlled Stage 1 local tracking experiment. Required inputs: reviewed frame/crop cache and exact local assets; see config and MODEL_INVENTORY.

ByteTrack uses supervision 0.25.1. BoT-SORT and StrongSORT are explicitly labeled BoxMOT 16.0.10 ports, not exact original-repository reproductions. McByte++ needs its isolated upstream/mask pipeline bridge. Freeze ONE low-threshold detection cache across methods, including missed/empty frames. No tracker may call a hidden detector. Trackers reset at reviewed cuts, gaps and new clips; track IDs are namespaced. Supply Re-ID weights for appearance methods. Official HOTA/IDF1 require dense, complete GT plus TrackEval; the runner provides supplemental purity, switches and anchor precision/coverage, not invented HOTA.

No results have been executed. GPU checks and opt-in flags precede model loading.

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
# Attach the unpacked laboratory as a Kaggle Dataset. Set LAB_ROOT if discovery is ambiguous.
LAB_ROOT = None
candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('README.md') if (p.parent/'scoutra_lab'/'common.py').is_file()}) if Path('/kaggle/input').exists() else []
if LAB_ROOT is None:
    if len(candidates) != 1: raise RuntimeError(f'Set LAB_ROOT to the attached laboratory folder. Candidates: {candidates}')
    LAB_ROOT = candidates[0]
LAB_ROOT = Path(LAB_ROOT)
assert (LAB_ROOT/'scoutra_lab'/'common.py').is_file()
sys.path.insert(0, str(LAB_ROOT))
WORK = Path('/kaggle/working/scoutra')
WORK.mkdir(parents=True, exist_ok=True)
print('Laboratory:', LAB_ROOT, 'Outputs:', WORK)


In [ ]:
INSTALL = False
PROFILE = 'tracking-reid.txt'
if INSTALL:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(LAB_ROOT/'configs'/'profiles'/PROFILE)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'check'])
    (WORK/'requirements-resolved.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
else:
    print('Installation skipped. Direct pins are provisional until pip check passes in this Kaggle image.')


## Check assets and device
Attach weights as a separate private Dataset only after reviewing the exact source/model terms. Populate `checkpoint.path`, expected SHA256, `license_reviewed`, and the raw class map where applicable. Directory checkpoints use the fingerprint defined in `common.asset_fingerprint`. The notebook does not download weights. Do not invent a Scoutra baseline when its checkpoint is unavailable.

One selected T4 runs one model. A second T4 may run an independent job after isolation/timing is verified. Never report concurrently contended runtime as an isolated benchmark. Initial runs use batch one; no automatic distributed model sharding.

In [ ]:
from scoutra_lab.common import load_config, environment
from scoutra_lab.pipelines import track
CFG = load_config(LAB_ROOT/'configs/tracker_bytetrack.yaml')
# Edit a copied config explicitly. Set a NEW experiment_id for every run.
# CFG['model']['checkpoint'] = {'path': 'VERIFIED_LOCAL_PATH', 'sha256': 'VERIFIED_SHA256', 'license_reviewed': True}
RUN_INFERENCE = False
CFG['execute'] = RUN_INFERENCE
print('Execution enabled:', CFG['execute'])
print('Method configuration:', CFG)
# environment() records package versions/device inventory without inference.
print(environment().get('devices', []))


In [ ]:
# With RUN_INFERENCE=False this creates a NOT EXECUTED record only.
# Requires base serialization dependencies; no model is loaded.
RESULT = track(CFG)
print('Result directory:', RESULT)
print((RESULT/'metrics.json').read_text())
print((RESULT/'runtime.json').read_text())


## Evaluation, runtime and error review
Open `metrics.json` only after checking execution/evaluation status. Keep quality on easy and hard conditions separate; compare matched thresholds and independent matches. Runtime synchronizes CUDA and records allocated/reserved peaks on the selected GPU; it excludes initial loading and includes disk/decode, preprocessing and serialization as specified in runtime.json. Add separate cold-load/warm steady-state measurements in later runs; no FPS value is prefilled.

Use `scoutra_lab.visualize.render` with the shared frames and result rows to save labeled overlays. Review identity-loss/switch timestamps against GT. Save selected-frame failures and manually reviewed error categories. Quantitative conclusions need confidence intervals and a practical margin; fewer than five independent matches remain descriptive. See EXPERIMENT_PLAN for preregistered progression gates.

## Exact implementations and official metrics
[supervision0.25.1 ByteTrack](https://github.com/roboflow/supervision/blob/0.25.1/supervision/tracker/byte_tracker/core.py); [BoxMOT16.0.10 distribution](https://pypi.org/project/boxmot/16.0.10/) BoT-SORT/StrongSORT ports (AGPL), which require separate OSNet assets. [McByte++ tracker](https://github.com/tstanczyk95/McBytePlusPlus/blob/be1bbc03f18e33e93e0a359bbcbfdfc4dc4ab6b9/yolox/tracker/mcbyteplusplus_tracker__with_reid.py) is a **blocked external method**, not implemented by the ByteTrack wrapper. Its masks/ReID must be provisioned and timed.

Use `evaluation.mot_export.export` for dense complete GT, then [original TrackEval12c8791](https://github.com/JonathonLuiten/TrackEval/tree/12c8791b303e0a0b50f753af204249e622d0281a) in the separate legacy CPU profile; `evaluation.run_trackeval` launches it. HOTA/IDF1 remain null until the official scorer actually runs.

Tracker timing includes initialization/reset and Re-ID loading per scene in the current runner. It differs from detector feature-model loading exclusions; inspect `runtime.json` before comparisons.

## Outputs and troubleshooting
Each new experiment ID creates `config.yaml`, `metadata.json`, `metrics.json`, `predictions.parquet`, `runtime.json`, `errors.json`, `report.md` and `visualizations/` below `/kaggle/working/scoutra/results/`. Existing IDs are never overwritten. Unexecuted predictions are an empty **schema container**, not measured empty detections; inspect status first. Missing annotations leave metrics null.

Download the entire results directory using Kaggle **Save Version → Save & Run All** only after flags are intentional, then download notebook outputs. Also save manifest, cache configuration, annotations, hashes and resolved environment. Do not enable all notebooks or all videos together.

- Missing `/kaggle/input/...`: print the mount contents and set the verified root. Dataset titles and mount paths can differ.
- No CUDA / unsupported GPU: stop; never silently move inference to this cloud or a local machine.
- Out of memory: reduce batch/frame/pixel budget in a new recorded configuration. The second T4 is not pooled memory.
- Missing/hash-mismatched weight: attach the exact reviewed checkpoint; do not auto-fetch a similarly named model.
- Dependency/ABI error: use a fresh Kaggle session and the method-specific profile. Preserve the failed `pip check`/freeze log. Do not upgrade all packages in place.
- Missing metrics: label and review the required frames. Visual inspection is not an accuracy result.
- Source/package API mismatch: preserve the error and version; do not patch the algorithm silently.

No execution or benchmark results are embedded in this notebook.
